# ML Enhancement: Market Basket Analysis (FP-Growth)

Discover products frequently purchased together using the FP-Growth algorithm.

## Business Value
- Cross-sell recommendations ("Customers also bought")
- Optimize store layouts based on co-purchase patterns
- Bundle pricing opportunities
- Promotion targeting with complementary products

## Data Flow
```
Configured Silver receipt tables --> FP-Growth --> Configured Gold association outputs
```

## Usage
Schedule this notebook on the cadence that fits your refresh requirements.

## Output
- Association rules with support, confidence, and lift are saved to the configured gold table
- Defaults are configurable via environment variables (support, confidence, receipt type filter, and top-N limit)

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.fpm import FPGrowth
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from datetime import datetime, timezone
import os
import mlflow


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="silver")
GOLD_DB = get_env("GOLD_DB", default="gold")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="market_basket")
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
RECEIPT_LINES_TABLE = get_env("RECEIPT_LINES_TABLE", default="fact_receipt_lines")
PRODUCT_ASSOCIATIONS_TABLE = get_env("PRODUCT_ASSOCIATIONS_TABLE", default="product_associations")
PRODUCT_RECOMMENDATIONS_TABLE = get_env("PRODUCT_RECOMMENDATIONS_TABLE", default="product_recommendations")
PRODUCT_ASSOCIATIONS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PRODUCT_ASSOCIATIONS_TABLE}"
PRODUCT_RECOMMENDATIONS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PRODUCT_RECOMMENDATIONS_TABLE}"
RECEIPT_TYPE_FILTER = get_env("RECEIPT_TYPE_FILTER", default="SALE")

# FP-Growth parameters from acceptance criteria
MIN_SUPPORT = float(get_env("MIN_SUPPORT", default="0.01"))  # 1% of transactions
MIN_CONFIDENCE = float(get_env("MIN_CONFIDENCE", default="0.3"))  # 30%
TOP_N_RULES = int(get_env("TOP_N_RULES", default="100"))  # Top 100 by lift



print(f"Configuration:")
print(f"  SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"  Source tables: {RECEIPTS_TABLE}, {RECEIPT_LINES_TABLE}")
print(f"  Receipt type filter: {RECEIPT_TYPE_FILTER}")
print(f"  Output tables: {PRODUCT_ASSOCIATIONS_TABLE_NAME}, {PRODUCT_RECOMMENDATIONS_TABLE_NAME}")
print(f"  MIN_SUPPORT={MIN_SUPPORT}, MIN_CONFIDENCE={MIN_CONFIDENCE}")
print(f"  TOP_N_RULES={TOP_N_RULES}")

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")

def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")

def save_gold(df, table_name):
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    row_count = spark.table(full_name).count()
    print(f"  {full_name}: {row_count} rows")
    return full_name, row_count

def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

ensure_database(GOLD_DB)

mlflow.set_experiment(EXPERIMENT_NAME)


In [ ]:
print("="*60)
print("MARKET BASKET ANALYSIS - TRANSACTION PREPARATION")
print("="*60)

In [ ]:
# Step 1: Prepare transaction baskets
# Join fact_receipts with fact_receipt_lines to get all items per receipt

if not silver_exists(RECEIPTS_TABLE) or not silver_exists(RECEIPT_LINES_TABLE):
    raise RuntimeError(
        f"Required tables {RECEIPTS_TABLE} and {RECEIPT_LINES_TABLE} not found in Silver"
    )

print("\nPreparing transaction baskets...")

# Read receipt headers using the configured receipt type filter
receipts = (
    read_silver(RECEIPTS_TABLE)
    .filter(F.col("receipt_type") == RECEIPT_TYPE_FILTER)
    .select("receipt_id_ext", "store_id", "event_ts")
)

# Read receipt lines
receipt_lines = (
    read_silver(RECEIPT_LINES_TABLE)
    .select("receipt_id_ext", "product_id")
)

# Join and create baskets (list of product IDs per receipt)
baskets = (
    receipts
    .join(receipt_lines, "receipt_id_ext")
    .groupBy("receipt_id_ext")
    .agg(
        F.collect_set("product_id").alias("items")
    )
    # Filter out single-item transactions (no associations possible)
    .filter(F.size(F.col("items")) > 1)
)

total_baskets = baskets.count()
print(f"  Total transaction baskets (multi-item): {total_baskets:,}")

if total_baskets == 0:
    raise RuntimeError("No multi-item transactions found. Cannot proceed with market basket analysis.")

In [ ]:
print("\n" + "="*60)
print("FP-GROWTH MODEL TRAINING")
print("="*60)

# Initialize FP-Growth model
fpGrowth = FPGrowth(
    itemsCol="items",
    minSupport=MIN_SUPPORT,
    minConfidence=MIN_CONFIDENCE
)

print(f"\nTraining FP-Growth model with {total_baskets:,} baskets...")
model = fpGrowth.fit(baskets)
print("  Model training complete")

In [ ]:
print("\n" + "="*60)
print("ASSOCIATION RULE EXTRACTION")
print("="*60)

# Extract frequent itemsets
frequent_itemsets = model.freqItemsets
frequent_itemsets_count = frequent_itemsets.count()
print(f"\nFrequent itemsets found: {frequent_itemsets_count:,}")

# Extract association rules
association_rules = model.associationRules
total_rules = association_rules.count()
print(f"Association rules found: {total_rules:,}")

if total_rules == 0:
    print("\nWARNING: No association rules found with current thresholds.")
    print("Consider lowering MIN_SUPPORT or MIN_CONFIDENCE parameters.")
    # Create empty result table
    from pyspark.sql.types import StructType, StructField, ArrayType, LongType, DoubleType, TimestampType
    schema = StructType([
        StructField("antecedent", ArrayType(LongType()), False),
        StructField("consequent", ArrayType(LongType()), False),
        StructField("support", DoubleType(), False),
        StructField("confidence", DoubleType(), False),
        StructField("lift", DoubleType(), False),
        StructField("computed_at", TimestampType(), False)
    ])
    result_df = spark.createDataFrame([], schema)
else:
    # Calculate lift and rank by it
    # Lift = confidence / (support of consequent)
    # Higher lift means stronger association
    
    # Get support for all items (consequent support needed for lift calculation)
    item_support = (
        frequent_itemsets
        .filter(F.size(F.col("items")) == 1)
        .select(
            F.col("items").getItem(0).alias("item"),
            F.col("freq").alias("item_freq")
        )
    )
    
    # Calculate lift for each rule
    rules_with_lift = (
        association_rules
        .withColumn("consequent_item", F.col("consequent").getItem(0))
        .join(
            item_support,
            F.col("consequent_item") == F.col("item"),
            "left"
        )
        .withColumn(
            "consequent_support",
            F.col("item_freq") / F.lit(total_baskets)
        )
        .withColumn(
            "lift",
            F.when(
                F.col("consequent_support") > 0,
                F.col("confidence") / F.col("consequent_support")
            ).otherwise(0.0)
        )
        .select(
            F.col("antecedent"),
            F.col("consequent"),
            # Support from the rule is actually the support of antecedent+consequent
            # We'll use confidence as the rule support for clarity
            F.col("confidence").alias("support"),
            F.col("confidence"),
            F.col("lift")
        )
    )
    
    # Get top N rules by lift
    top_rules = (
        rules_with_lift
        .orderBy(F.desc("lift"))
        .limit(TOP_N_RULES)
        .withColumn(
            "computed_at",
            F.lit(datetime.now(timezone.utc))
        )
    )
    
    result_df = top_rules
    print(f"\nTop {TOP_N_RULES} association rules by lift prepared")

In [ ]:
print("\n" + "="*60)
print("SAVING TO GOLD LAYER")
print("="*60)

# Save association rules to the configured gold table
rules_table, rules_saved_count = save_gold(result_df, PRODUCT_ASSOCIATIONS_TABLE)
saved_rules_df = spark.table(rules_table)

print("\nSample association rules (top 10 by lift):")
if saved_rules_df.limit(1).count() > 0:
    saved_rules_df.orderBy(F.desc("lift")).limit(10).show(truncate=False)
else:
    print("  No rules to display")

In [ ]:
print("\n" + "="*60)
print("CREATING PRODUCT RECOMMENDATION VIEW")
print("="*60)

# Create a table that makes recommendations easier to query
# For a given product, what products are frequently bought with it?

rules_df = spark.table(PRODUCT_ASSOCIATIONS_TABLE_NAME)

if rules_df.limit(1).count() > 0:
    recommendations = (
        rules_df
        .withColumn("antecedent_product", F.explode("antecedent"))
        .withColumn("consequent_product", F.explode("consequent"))
        .select(
            F.col("antecedent_product").alias("product_id"),
            F.col("consequent_product").alias("recommended_product_id"),
            "support",
            "confidence",
            "lift",
            "computed_at"
        )
    )
    
    recommendations_table, recommendations_count = save_gold(recommendations, PRODUCT_RECOMMENDATIONS_TABLE)
    
    print("\nSample recommendations (top 10):")
    (
        spark.table(recommendations_table)
        .orderBy("product_id", F.desc("lift"))
        .limit(10)
        .show(truncate=False)
    )
else:
    print("  Skipping: no rules to create recommendations")

In [ ]:
# Log results to MLflow
with mlflow.start_run(
    run_name=f"fpgrowth_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    mlflow.log_params({
        "algorithm": "fpgrowth",
        "min_support": MIN_SUPPORT,
        "min_confidence": MIN_CONFIDENCE,
        "top_n_rules": TOP_N_RULES,
    })

    rules_count = spark.table(PRODUCT_ASSOCIATIONS_TABLE_NAME).count()
    mlflow.log_metrics({
        "rules_saved": rules_count,
    })

    print(f"MLflow run: {run.info.run_id}")
    print(f"Rules logged: {rules_count}")


In [ ]:
print("\n" + "="*60)
print("MARKET BASKET ANALYSIS COMPLETE")
print("="*60)

# Summary statistics
rules_saved_count = spark.table(PRODUCT_ASSOCIATIONS_TABLE_NAME).count()
recommendations_count = (
    spark.table(PRODUCT_RECOMMENDATIONS_TABLE_NAME).count()
    if spark.catalog.tableExists(PRODUCT_RECOMMENDATIONS_TABLE_NAME)
    else 0
)
print(f"\nSummary:")
print(f"  Transaction baskets analyzed: {total_baskets:,}")
print(f"  Frequent itemsets: {frequent_itemsets_count:,}")
print(f"  Association rules (all): {total_rules:,}")
print(f"  Top rules saved: {rules_saved_count}")
print(f"  Product recommendations saved: {recommendations_count}")

# Show Gold tables
gold_tables = spark.sql(f"SHOW TABLES IN {LAKEHOUSE_NAME}.{GOLD_DB}").collect()
print(f"\nGold ({GOLD_DB}): {len(gold_tables)} tables")